# LR(1) and LALR State Merging

F# counterpart of `Python/Chapter-10/Mysterious-Conflicts.ipynb`. Build once with `python FSharp/tools/build.py`, then run this notebook from top to bottom with a fresh .NET (F#) kernel. The scanner and parser are generated from real FsLexYacc specifications.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "Lalr/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "Lalr/bin/Debug/net9.0/Lalr.dll"
#load "../Support/Visualization.fsx"
open System
open System.IO
open FormalLanguages.Lalr
open Visualization

## Typed syntax and scanner

The semantic actions construct a closed expression type. Numeric tokens are converted once, in the scanner; arithmetic entry points require integer literals, and the symbolic calculator accepts decimal and exponent forms; punctuation becomes a token constructor. Newlines advance the source position, and illegal input raises a positioned error.

```fsharp
namespace FormalLanguages.Lalr

/// A closed syntax tree makes every interpreter case visible to the compiler.
type Expression =
    | Number of float
    | Name of string
    | Unary of string * Expression
    | Binary of string * Expression * Expression

type Statement =
    | Assign of string * Expression
    | Evaluate of Expression

type Lexeme =
    | Numeric of float
    | Identifier of string
    | Operator of string
    | Character of char
    | End
```

The shared `.fsl` file supplies an expression mode and a character mode:

```text
{
module FormalLanguages.Lalr.Lexer
open System.Globalization
open FSharp.Text.Lexing
}
let digit = ['0'-'9']
let letter = ['a'-'z' 'A'-'Z']
let number = ('0' | ['1'-'9'] digit*) ('.' digit+)? (['e' 'E'] ['+' '-']? digit+)?
rule token = parse
| [' ' '\t' '\r']+ { token lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
| number { Numeric (System.Double.Parse(LexBuffer<char>.LexemeString lexbuf, CultureInfo.InvariantCulture)) }
| letter (letter | digit | ['_' '<' '>'])* { Identifier (LexBuffer<char>.LexemeString lexbuf) }
| ":=" { Operator ":=" }
| ['+' '-' '*' '/' '^' '(' ')' ';' '↔' '→' '∧' '∨' '¬' '⊤' '⊥'] { Operator (LexBuffer<char>.LexemeString lexbuf) }
| eof { End }
| _ { failwithf "Illegal character %A at line %d, column %d" (LexBuffer<char>.LexemeString lexbuf) lexbuf.StartPos.Line (lexbuf.StartPos.Column + 1) }
and character = parse
| [' ' '\t' '\r']+ { character lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; character lexbuf }
| ['U' 'V' 'W' 'X' 'v' 'w' 'x' 'y' 'z'] { Character (lexbuf.LexemeChar 0) }
| eof { End }
| _ { failwithf "Illegal character %A" (LexBuffer<char>.LexemeString lexbuf) }
```

## Parser specification

FsYacc generates token constructors and parse tables from this `.fsy` specification. `$1`, `$2`, and `$3` refer to values on the right-hand side. `main` consumes `EOF`, so a valid prefix with trailing junk is rejected.

```text
%{
open FormalLanguages.Lalr
%}
%token U V W X Y Z EOF
%start main
%type <string> main
%%
main: s EOF { $1 }
s: V a Y { "vay" } | W b Y { "wby" } | V b Z { "vbz" } | W a Z { "waz" }
a: X { () }
b: X { () }
```

In [ ]:
printfn "%s" (File.ReadAllText "Lalr/Mysterious.fsyacc.output")

In [ ]:
let attempt parse text =
    try Ok (parse text)
    with error -> Error error.Message

The grammar is LR(1) but not LALR(1). Merging states with the same LR(0) core combines incompatible lookahead sets. FsYacc reports two reduce/reduce conflicts; the chosen production accepts `vxy` and `wxz` but loses `vxz` and `wxy`. This is measured behavior of FsLexYacc 11.3.0, rather than a claim about PLY’s implementation.

In [ ]:
let results = ["vxy"; "wxy"; "vxz"; "wxz"] |> List.map (fun text -> text, attempt Parse.mysterious text)
checkNotebook (results[0] |> snd = Ok "vay")
checkNotebook (results[3] |> snd = Ok "waz")
checkNotebook (results[1] |> snd |> Result.isError)
checkNotebook (results[2] |> snd |> Result.isError)
results